# Step 4: Model Training

Train a **RandomForest classifier** using pre-computed features and register in Snowflake Model Registry.

## Features Used

The model is trained on **raw features + computed features** from preprocessing:
- **Raw**: Age, BMI, Heart Rate, Blood Pressure, Lab Values, etc.
- **Computed**: SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY

## Snowflake Services Used

| Service | Purpose |
|---------|---------|
| **ML Jobs** | Remote training on SPCS compute pools |
| **Model Registry** | Version and store models |

## Prerequisites

- Run notebooks 01-03 first (03 creates TRAINING_FEATURES and TEST_FEATURES tables)

## Imports and Configuration

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

In [ ]:
COMPUTE_POOL = config.compute.compute_pool

pool_exists = session.sql(f"SHOW COMPUTE POOLS LIKE '{COMPUTE_POOL}'").collect()
if not pool_exists:
    session.sql(f"""
        CREATE COMPUTE POOL {COMPUTE_POOL}
        MIN_NODES = 1 MAX_NODES = 1
        INSTANCE_FAMILY = CPU_X64_S
        AUTO_SUSPEND_SECS = 300
    """).collect()
    print(f"Created compute pool: {COMPUTE_POOL}")
else:
    print(f"Compute pool exists: {COMPUTE_POOL}")

# ML Jobs: Remote Model Training

In [ ]:
from snowflake.ml.jobs import submit_directory

job = submit_directory(
    dir_path="source",
    entrypoint="train.py",
    compute_pool=COMPUTE_POOL,
    stage_name=f"{DB}.{SCHEMA}.JOB_PAYLOADS",
    env_vars={},
    pip_requirements=[]
)
print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")

In [ ]:
print("Waiting for job to complete...")
job.wait()

print(f"\nFinal status: {job.status}")
if job.status == "DONE":
    print("\n=== Job Logs ===")
    logs = job.get_logs()
    print(logs[-3000:] if len(logs) > 3000 else logs)
else:
    print(f"Job failed: {job.status}")
    print(job.get_logs())

## Evaluate Model

In [ ]:
from snowflake.ml.registry import Registry
from source.utils import get_feature_config

model_name = "PATIENT_RISK_MODEL"
test_table = f"{DB}.{SCHEMA}.TEST_FEATURES"
feature_config = get_feature_config(config)
numeric_columns = feature_config["all_numeric_features"]
categorical_columns = feature_config["all_categorical_features"]
feature_columns = numeric_columns + categorical_columns
target_column = feature_config["target_column"]

# == Get Model from Registry ==
registry = Registry(
            session,
            database_name=DB,
            schema_name=SCHEMA,
        )
model = registry.get_model(model_name)
model_version_obj = model.versions()[-1]


# == Get Test Data ==
logger.info(f"Loading test data from {test_table}")
test_df = session.table(test_table).to_pandas()
test_df.columns = [c.upper() for c in test_df.columns]
features = test_df[feature_columns]
y_true = test_df[target_column].values


logger.info("Running inference")
predictions_df = model_version_obj.run(features, function_name="predict")
y_pred = predictions_df["output_feature_0"].values


proba_df = model_version_obj.run(features, function_name="predict_proba")
y_proba = proba_df.values

In [ ]:
import uuid
import pandas as pd

records = []
metrics_table = f"{DB}.{SCHEMA}.MODEL_METRICS"
for metric_name, metric_value in metrics.items():
    if metric_name in ["confusion_matrix", "per_class", "evaluated_at"]:
        continue

    if isinstance(metric_value, (int, float)):
        records.append(
            {
                "METRIC_ID": f"M{uuid.uuid4().hex[:8].upper()}",
                "MODEL_NAME": model_name or metrics.get("model_name"),
                "MODEL_VERSION": model_version_obj.version_name,
                "METRIC_NAME": metric_name,
                "METRIC_VALUE": float(metric_value),
                "METRIC_DETAILS": None,
                "EVALUATED_AT": datetime.now(),
            }
        )

if records:
    df = pd.DataFrame(records)
    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("append").save_as_table(metrics_table)
    logger.info(f"Logged {len(records)} metrics to {metrics_table}")

session.table(metrics_table).to_pandas()